<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/DerivedTables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Derived Tables

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 142 (delta 37), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 11.20 MiB | 3.98 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [2]:
!wget -O CORPUS.csv "https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"
!wget -O VOCAB.csv "https://virginia.box.com/shared/static/a2njs9ipjxrgam9un7sr4yn3f6f196ny.csv"

--2026-04-14 21:30:21--  https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-14 21:30:22--  https://virginia.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-14 21:30:22--  https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443.

In [3]:
import pandas as pd
CORPUS = pd.read_csv('CORPUS.csv', delimiter='|')
LIB = pd.read_csv('LIB.csv', delimiter='|')
VOCAB = pd.read_csv('VOCAB.csv', delimiter='|')

## BOW

In [4]:
OHCO = ['Artist', 'Album', 'Title', 'token_str']
bags = dict(
    TITLES = OHCO[:3],
    ALBUMS = OHCO[:2],
    ARTISTS = OHCO[:1]
)

In [5]:
def create_bow(CORPUS, bag):
    bow = (
        CORPUS
        .groupby(bags[bag]+['term_str'])
        .term_str.count()
        .to_frame('n')
    )
    return bow

In [6]:
BOW = create_bow(CORPUS, bag='TITLES')
BOW.head()

n
Artist        Album                                 Title         term_str    
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You a         18
                                                                  all        6
                                                                  and        9
                                                                  around     2
                                                                  at         2

In [7]:
print('Number of observations:', len(BOW))

Number of observations: 235266


In [8]:
BOW.to_csv('/content/DS5001-Final-Project/BOW.csv', sep='|', index=True)

## DTM

In [9]:
DTM = BOW.n.unstack(fill_value=0)
DTM.head()

term_str                                                                        a  \
Artist        Album                                 Title                           
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              18   
              Ariana Grande                         Ariana Grande Tour Guide    0   
                                                    I’m Every Woman/Vogue      14   
                                                    Leave Me Lonely (Reprise)   5   
                                                    Love Me Harder/breathin     2   

term_str                                                                       all  \
Artist        Album                                 Title                            
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                6   
              Ariana Grande                         Ariana Grande Tour Guide     0   
                                                    I’m Every Woman/Vogue        8   
                                                    Leave Me Lonely (Reprise)    0   
                                                    Love Me Harder/breathin      3   

term_str                                                                       and  \
Artist        Album                                 Title                            
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                9   
              Ariana Grande                         Ariana Grande Tour Guide     0   
                                                    I’m Every Woman/Vogue        6   
                                                    Leave Me Lonely (Reprise)    1   
                                                    Love Me Harder/breathin     32   

term_str                                                                       around  \
Artist        Album                                 Title                               
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                   2   
              Ariana Grande                         Ariana Grande Tour Guide        0   
                                                    I’m Every Woman/Vogue           2   
                                                    Leave Me Lonely (Reprise)       0   
                                                    Love Me Harder/breathin         0   

term_str                                                                       at  \
Artist        Album                                 Title                           
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You               2   
              Ariana Grande                         Ariana Grande Tour Guide    0   
                                                    I’m Every Woman/Vogue       0   
                                                    Leave Me Lonely (Reprise)   0   
                                                    Love Me Harder/breathin     0   

term_str                                                                       away  \
Artist        Album                                 Title                             
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                 1   
              Ariana Grande                         Ariana Grande Tour Guide      0   
                                                    I’m Every Woman/Vogue         1   
                                                    Leave Me Lonely (Reprise)     1   
                                                    Love Me Harder/breathin       1   

term_str                                                                       be  \
Artist        Album                                 Title                           
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You               2   
              Ariana Grande                         Ariana Grande Tour Guide    0   
                            

In [10]:
DTM.to_csv('/content/DS5001-Final-Project/DTM.csv', sep='|', index=True)

## TFIDF

In [11]:
import numpy as np

def get_tfidf(DTM, tf_method='sum', idf_method='standard', item_type='term_str'):

    if tf_method == 'sum':
        TF = (DTM.T / DTM.T.sum()).T
    elif tf_method == 'smooth':
        TF = (DTM.T / DTM.T.sum()).T + 1
    elif tf_method == 'max':
        TF = (DTM.T / DTM.T.max()).T
    elif tf_method == 'log':
        TF = (np.log2(1 + DTM.T)).T
    elif tf_method == 'raw':
        TF = DTM
    elif tf_method == 'bool':
        TF = DTM.astype('bool').astype('int')
    else:
        raise ValueError(f"TF method {tf_method} not found.")

    DF = DTM.astype('bool').sum()
    N = len(DTM)

    if idf_method == 'standard':
        IDF = np.log2(N / DF)
    elif idf_method == 'max':
        IDF = np.log2(DF.max() / DF)
    elif idf_method == 'plus':
        IDF = np.log2(N / DF) + 1
    elif idf_method == 'smooth':
        IDF = np.log2((1 + N) / (1 + DF)) + 1
    else:
        raise ValueError(f"IDF method {idf_method} not found.")

    TFIDF = TF * IDF

    return TFIDF

In [12]:
TFIDF = get_tfidf(DTM)
TFIDF.head()

term_str                                                                              a  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.014640   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.007248   
                                                    Leave Me Lonely (Reprise)  0.018010   
                                                    Love Me Harder/breathin    0.001159   

term_str                                                                            all  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.013352   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.011332   
                                                    Leave Me Lonely (Reprise)  0.000000   
                                                    Love Me Harder/breathin    0.004758   

term_str                                                                            and  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.004929   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.002092   
                                                    Leave Me Lonely (Reprise)  0.002425   
                                                    Love Me Harder/breathin    0.012488   

term_str                                                                         around  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.015389   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.009796   
                                                    Leave Me Lonely (Reprise)  0.000000   
                                                    Love Me Harder/breathin    0.000000   

term_str                                                                             at  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.010132   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.000000   
                                                    Leave Me Lonely (Reprise)  0.000000   
                                                    Love Me Harder/breathin    0.000000   

term_str                                                                           away  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.007574   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.004821   
                                                    Leave Me Lonely (Reprise)  0.033541   
                                                    Love Me Harder/breathin    0.005397   

term_str                                                                             be  \
Artist        Album                                 Title                        

In [13]:
TFIDF.to_csv('/content/DS5001-Final-Project/TFIDF.csv', sep='|', index=True)

## TFIDF_L2

In [14]:
from numpy.linalg import norm

TFIDF_L2 = TFIDF.apply(lambda x: x / norm(x), 1)

# Keep only words that appear in at least 2 documents
DF = DTM.astype('bool').sum()
TFIDF_L2 = TFIDF_L2.loc[:, DF >= 2]

TFIDF_L2.head()

term_str                                                                              a  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.022310   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.020322   
                                                    Leave Me Lonely (Reprise)  0.013772   
                                                    Love Me Harder/breathin    0.001267   

term_str                                                                            all  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.020347   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.031773   
                                                    Leave Me Lonely (Reprise)  0.000000   
                                                    Love Me Harder/breathin    0.005202   

term_str                                                                            and  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.007511   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.005864   
                                                    Leave Me Lonely (Reprise)  0.001854   
                                                    Love Me Harder/breathin    0.013654   

term_str                                                                         around  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.023451   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.027465   
                                                    Leave Me Lonely (Reprise)  0.000000   
                                                    Love Me Harder/breathin    0.000000   

term_str                                                                             at  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.015441   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.000000   
                                                    Leave Me Lonely (Reprise)  0.000000   
                                                    Love Me Harder/breathin    0.000000   

term_str                                                                           away  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.011542   
              Ariana Grande                         Ariana Grande Tour Guide   0.000000   
                                                    I’m Every Woman/Vogue      0.013517   
                                                    Leave Me Lonely (Reprise)  0.025648   
                                                    Love Me Harder/breathin    0.005901   

term_str                                                                             be  \
Artist        Album                                 Title                        

In [15]:
print("Number of features:", TFIDF_L2.shape[1])

Number of features: 8350


In [16]:
TFIDF_L2.to_csv('/content/DS5001-Final-Project/TFIDF_L2.csv', sep='|', index=True)